# Cleaning 2.3 - Clean energy calculations

This notebook does the following:
    (1) Creates variables so that the dataset can be merged with the equipment calculators
    (2) Merges the dataset with the equipment calculators
    (3) Performs the energy calculations

## Set-up

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
import re
from pathlib import Path
CODE_ROOT = Path.cwd().parents[0]
sys.path.append(str(CODE_ROOT))
import config
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
import os
from fill_missing_mode import fill_with_equipment_mode
from assign_set_temp import assign_set_temp

In [2]:
# Load data
equipment = pd.read_csv(
    config.PROCESSED_DATA / "panel_processed_2.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

In [3]:
# Load labs data
labs = pd.read_csv(
    config.PROCESSED_DATA / "individual_processed_4.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

In [4]:
# Load equipment and fc calculations
equipment_calculations = pd.read_csv(
    config.SPARK_DATA / "2_Clean" / "equipment_calculations.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

fc_calculations = pd.read_csv(
    config.SPARK_DATA / "2_Clean" / "fc_calculations.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

In [5]:
# Load data dictionaries
equipment_data_dict = pd.read_excel(config.DATA_DICTIONARIES / "data_dictionary.xlsx", sheet_name="Equipment")

In [6]:
# Merge equipment with labs to get institute
equipment = equipment.merge(labs[["labgroupid", "institute"]], left_on = "labgroupid", right_on = "labgroupid", how = "left")

## (1) Prepare for merging with equipment calculators

### (1a) Generic variables (number, hours, days, door_openings)

In [7]:
# Clean number variable

# If number is missing, set number to 1
equipment.loc[equipment["number"].isna(), "number"] = 1

# Make number integer
equipment["number"] = equipment["number"].astype(int)


In [8]:
# Clean hours and days variables
print(equipment["equipment"].unique())

#Add mask for equipment that has hours and days 
has_day_hours_mask = equipment["equipment"].apply(lambda x: any(val in x for val in ("fc", "heater", "bath", "cryostat", "microbio", "glassware", "it")))
#always_on = ("fridge", "freezer", "ult") #Included for completeness

# If hours is missing or "Unknown", set hours to 9
equipment.loc[(has_day_hours_mask & (equipment["hours"].isna()) | (equipment["hours"] == "Unknown")), "hours"] = 9

# If days is missing or "Unknown", set days to 255
equipment.loc[(has_day_hours_mask & (equipment["days"].isna()) | (equipment["days"] == "Unknown")), "days"] = 255

# Make hours and days numeric
equipment["hours"] = pd.to_numeric(equipment["hours"], errors="raise")
equipment["days"] = pd.to_numeric(equipment["days"], errors="raise")

['freezer' 'heater' 'it' 'bath' 'fc' 'fridge' 'microbio' 'ult' 'cryostat'
 'glassware' 'incubator']


In [9]:
# Clean door openings (note: door opening is now in minutes!, and this impacts fridge, freezer, and ults)

# Mask for equipment with door openings (fridge, freezer, ult)
has_door_opening_mask = equipment["equipment"].apply(lambda x: any(val in x for val in ("freezer", "fridge", "ult")))

# Check unique values for door openings
print("Unique values in door_openings before cleaning:")
print(equipment.loc[has_door_opening_mask, "door_openings"].unique())

# If door openings is unknown or NaN, set to 0
equipment.loc[
    (has_door_opening_mask & 
     (equipment["door_openings"].isna() | equipment["door_openings"].isin(["Unknown"]))), "door_openings"
] = 0

# Make door openings numeric
equipment["door_openings"] = pd.to_numeric(equipment["door_openings"], errors="raise")

Unique values in door_openings before cleaning:
[nan '0' '1' '2' '0.03' '5' '10' '15' '20' '3' '6' '30' '12' 'Unknown' '7'
 '4' '100' '0.3' '0.01' '7.5' '40' '60' '27' '0.1' '0.8' '8' '25' '0.5'
 '10.0' '0.05' '0.06' '2.5' '45' '3.0' '0.0' '42' '1.5']


In [10]:
# 1. Fix all data within its respective starting column

### (1b) FC variables (hours_open, controller_type, sash_width, lifted, face_velocity)

In [11]:
# Clean FC variables

# Check unique values in hours_open
print("Unique values in hours_open before cleaning:")
print(equipment[equipment["equipment"] == "fc"]["hours_open"].unique())

# Fill missing/unknown values for hours_open with mode
equipment = fill_with_equipment_mode(
    equipment,
    value_col="hours_open",
    equipment_value="fc",
)

# Make hours_open numeric
equipment["hours_open"] = pd.to_numeric(equipment["hours_open"], errors="raise")

Unique values in hours_open before cleaning:
[ 1.  2.  3. nan  8.  6.  4.  5.  0. 24.]


In [12]:
# Controller type

# Check unique values and distribution of controller_type
print(equipment[equipment["equipment"] == "fc"]["controller_type"].unique())
print(equipment[equipment["equipment"] == "fc"]["controller_type"].value_counts(dropna=False))

# Clean to shortened names ("VAV"/"CAV" instead of "Variable Air Volume (VAV)"/"Constant Air Volume (CAV)")
equipment["controller_type"] = equipment["controller_type"].replace({
    "Variable Air Volume (VAV)": "VAV",
    "Constant Air Volume (CAV)": "CAV"
})

# Fill missing/unknown values for controller_type with "VAV"
equipment.loc[
    (equipment["equipment"] == "fc") & 
    (equipment["controller_type"].isna() | equipment["controller_type"].isin(["Unknown"])),
    "controller_type"
] = "VAV"

print(equipment[equipment["equipment"] == "fc"]["controller_type"].unique())


['Variable Air Volume (VAV)' 'Constant Air Volume (CAV)' nan 'Unknown']
controller_type
Variable Air Volume (VAV)    108
Constant Air Volume (CAV)     64
NaN                           10
Unknown                       10
Name: count, dtype: int64
['VAV' 'CAV']


In [13]:
# Sash width (convert width to meters and numeric)

# Check unique values for sash_width
print(equipment[equipment["equipment"] == "fc"]["sash_width"].unique())

# Set sash_width to the mode within each institute, if missing or unknown
equipment = fill_with_equipment_mode(
    equipment,
    value_col="sash_width",
    equipment_value="fc",
    groupby_cols="institute"
)

# If missing or unknown, set to 1500mm
# equipment.loc[
#     (equipment["equipment"] == "fc") & 
#     (equipment["sash_width"].isna() | equipment["sash_width"].isin(["Unknown"])),
#     "sash_width"
# ] = 1500

# Set to numeric and convert from mm to m
equipment["sash_width"] = pd.to_numeric(equipment["sash_width"], errors="raise")/1000

# Check that all values are within 0.5-3m
assert equipment.loc[equipment["equipment"] == "fc", "sash_width"].between(0.5, 3).all(), "Some sash_width values are outside the range of0.5-3 meters."

['1700' '1080' '1100' '1200' '1500' nan 'Unknown' '1200.0' '1070' '1350'
 '1050' '1150' '1450' '1000' '900' '1170' '1110' '2000' '1300' '1250'
 '1190' '1160' '500' '1120' '1900']


In [14]:
# Face velocity

# Print unique values and distribution of face_velocity
print(equipment[equipment["equipment"] == "fc"]["face_velocity"].unique())

# Create indicators for being in m3/h or m/s
equipment.loc[
    (equipment["equipment"] == "fc") &
    (equipment["face_velocity"].str.contains("m3/h", na=False)),
    "face_velocity_unit_m3/h"
] = True
equipment.loc[
    (equipment["equipment"] == "fc") &
    (equipment["face_velocity"].str.contains("m/s", na=False)),
    "face_velocity_unit_m/s"
] = True

# Create variable which is in m3/h (minus " m3/h" from the string and convert to numeric)
equipment.loc[
    (equipment["equipment"] == "fc") & 
    (equipment["face_velocity_unit_m3/h"] == True),
    "face_velocity_m3/h"
] = equipment["face_velocity"].str.replace("m3/h", "")
equipment["face_velocity_m3/h"] = pd.to_numeric(equipment["face_velocity_m3/h"], errors="raise")

# Create variable which is in m/s (minus " m/s" from the string and convert to numeric)
equipment.loc[
    (equipment["equipment"] == "fc") &
    (equipment["face_velocity_unit_m/s"] == True),
    "face_velocity_m/s"
] = equipment["face_velocity"].str.replace("m/s", "")
equipment["face_velocity_m/s"] = pd.to_numeric(equipment["face_velocity_m/s"], errors="raise")

# For entries "Zentrum", "Irchel", "Unknown", missing, or in m3/h, set face_velocity_m/s to 0.2
equipment.loc[
    (equipment["equipment"] == "fc") &
    (equipment["face_velocity"].isin(["Zentrum", "Irchel", "Unknown"]) 
     | equipment["face_velocity"].isna() 
     | equipment["face_velocity_unit_m3/h"] == True),
    "face_velocity_m/s"
] = 0.2

['280m3/h' '1m/s' nan 'Unknown' '360m3/h' '450m3/h' '325m3/h' '0.5m/s'
 '0.385m/s' '0.397m/s' '400m3/h' '480m3/h' '600m3/h' '0.13m/s' '233m3/h'
 '235m3/h' '260m3/h' '0.35m/s' '340m3/h' 'Irchel' '0.118m/s' '0.4m/s'
 '0.45m/s' '0.6m/s' '396m3/h' '265m3/h' '470m3/h' 'Zentrum']


### (1c) Fridge variables (size_fridge)

In [15]:
# Size fridge

# Check unique values for size_fridge and distribution 
print(equipment[equipment["equipment"] == "fridge"]["size_fridge"].unique())

# Before splitting up, replace any "Unknown" or missing with the mode
equipment = fill_with_equipment_mode(
    equipment,
    value_col="size_fridge",
    equipment_value="fridge"
)

# Create variable size_fridge_1 which is "Fan"/"Convection"
equipment.loc[
    (equipment["equipment"] == "fridge") &
    (equipment["size_fridge"].str.contains("fan", case=False, na=False)),
    "size_fridge_1"
] = "Fan"
equipment.loc[
    (equipment["equipment"] == "fridge") &
    (equipment["size_fridge"].str.contains("convection", case=False, na=False)),
    "size_fridge_1"
] = "Convection"

# Create variable size_fridge_2 which extracts the size range (e.g. 250-425L) from size_fridge, and fix the range 250-425 to 250-450
equipment.loc[
    (equipment["equipment"] == "fridge")
    & (equipment["size_fridge"].str.contains("\\(~", na=False)),
    "size_fridge_2"
] = equipment["size_fridge"].str.split("\\(~").str[1].str.split("\\)").str[0].replace({"250-425L": "250-450L"})

#Note fridge has no door_opening option on website, but it still is in the code and was there previously

['Under bench fan assisted (~100-160L)'
 'Under bench convection (~100-160L)'
 'Tall single door fan assisted (~250-425L)'
 'Tall single door fan assisted (~500-700L)'
 'Tall single door convection (~250-425L)'
 'Under bench fan assisted (~250-425L)'
 'Tall double door fan assisted (~1200-1500L)'
 'Tall single door convection (~100-160L)']


### (1d) Freezer variables (size_freezer, temp_freezer, icing, refrigerant)

In [17]:
# Freezer size

# Check unique values for size_freezer and distribution
print(equipment[equipment["equipment"] == "freezer"]["size_freezer"].unique())
print(equipment[equipment["equipment"] == "freezer"]["size_freezer"].value_counts(dropna=False))

# Before splitting up, replace any "Unknown" or missing with the mode
equipment = fill_with_equipment_mode(
    equipment,
    value_col="size_freezer",
    equipment_value="freezer"
)

# Create variable size_freezer_1 which is Chest/Under Bench/Upright
equipment["size_freezer_1"] = equipment["size_freezer"]
equipment["size_freezer_1"] = equipment["size_freezer_1"].replace({
    "Tall and slim single door (~190-38L)": "Upright",
    "Tall and wide single door (~390-525L)": "Upright",
    "Chest (~390-525L)": "Chest",
    "Chest (~190-380L)": "Chest",
    "Under bench (~80-130L)": "Under Bench"
})

# Create variable size_freezer_2 which extracts the size range from size_freezer, and fixes the range 190-38 to 190-380, and 80-130 to 90-130
equipment.loc[
    (equipment["equipment"] == "freezer")
    & (equipment["size_freezer"].str.contains("\\(~", na=False)),
    "size_freezer_2"
] = equipment["size_freezer"].str.split("\\(~").str[1].str.split("\\)").str[0].replace(
    {"190-38L": "190-380L", "80-130L": "90-130L"}
)

print(equipment[equipment["equipment"] == "freezer"]["size_freezer_1"].unique())
print(equipment[equipment["equipment"] == "freezer"]["size_freezer_2"].unique())

['Tall and slim single door (~190-38L)' 'Under bench (~80-130L)'
 'Tall and wide single door (~390-525L)' 'Chest (~390-525L)'
 'Chest (~190-380L)']
size_freezer
Tall and slim single door (~190-38L)     121
Under bench (~80-130L)                   103
Tall and wide single door (~390-525L)     46
Chest (~390-525L)                         15
Chest (~190-380L)                         12
Name: count, dtype: int64
['Upright' 'Under Bench' 'Chest']
['190-380L' '90-130L' '390-525L']


In [18]:
# Freezer temp

# Check unique values
print(equipment[equipment["equipment"] == "freezer"]["temp_freezer"].unique())

# Replace any "Unknown" or missing with the mode
equipment = fill_with_equipment_mode(
    equipment,
    value_col="temp_freezer",
    equipment_value="freezer"
)

# Convert to numeric
equipment["temp_freezer"] = pd.to_numeric(equipment["temp_freezer"], errors="raise")

# Bound freezer temperatures to -30 to -20 (anything below -30 becomes -30, anything above -20 becomes -20)
equipment.loc[
    (equipment["equipment"] == "freezer") & (equipment["temp_freezer"] < -30),
    "temp_freezer"
] = -30
equipment.loc[
    (equipment["equipment"] == "freezer") & (equipment["temp_freezer"] > -20),
    "temp_freezer"
] = -20

[nan '-20' '-21' '-26' '-30' '-22' '-25' '-32' '-24' '-20.0' '-18' '-28'
 'Unknown']


In [21]:
# Icing

# Check unique values for icing and distribution
print(equipment[equipment["equipment"] == "freezer"]["icing"].unique())
print(equipment[equipment["equipment"] == "freezer"]["icing"].value_counts(dropna=False))

# Combine "No" and "A little bit" into "No, clear of ice", and "Yes" into "Yes, iced up"
equipment["icing"] = equipment["icing"].replace({
    "No icing": "No, clear of ice",
    "A little building on the drawers and shelves": "No, clear of ice",
    "Yes it's ≥2mm thick and can make opening the door/accessing contents challenging": "Yes, iced up"
})

# Replace any "Unknown" or missing with the mode
equipment = fill_with_equipment_mode(
    equipment,
    value_col="icing",
    equipment_value="freezer"
)

[nan 'No icing' 'A little building on the drawers and shelves'
 "Yes it's ≥2mm thick and can make opening the door/accessing contents challenging"]
icing
A little building on the drawers and shelves                                        148
No icing                                                                            109
Yes it's ≥2mm thick and can make opening the door/accessing contents challenging     36
NaN                                                                                   4
Name: count, dtype: int64


In [22]:
# Refrigerant

print(equipment[equipment["equipment"] == "freezer"]["refrigerant"].unique())
print(equipment[equipment["equipment"] == "freezer"]["refrigerant"].value_counts(dropna=False))

# Replace any "Unknown" or missing with the mode
equipment = fill_with_equipment_mode(
    equipment,
    value_col="refrigerant",
    equipment_value="freezer"
)

[nan 'HCs (usually more modern units)' 'HFCs (usually an older unit)'
 'Unknown']
refrigerant
HCs (usually more modern units)    192
HFCs (usually an older unit)        90
NaN                                  8
Unknown                              7
Name: count, dtype: int64


In [ ]:
# Drawers (Note: Drawer type has switched from Yes+Plastic/NoDrawers/No+Wire to Plastic/Wire)
print(equipment[equipment["equipment"] == "freezer"]["drawers"].unique())
print(equipment[equipment["equipment"] == "freezer"]["drawers"].value_counts(dropna=False))

# Replace "Unknown" or missing with mode
equipment = fill_with_equipment_mode(
    equipment,
    value_col="drawers",
    equipment_value="freezer"
)

['Yes they are solid plastic' 'Yes they are made from plastic coated wire'
 'No there are no drawers or most are missing']
drawers
Yes they are solid plastic                     248
No there are no drawers or most are missing     30
Yes they are made from plastic coated wire      19
Name: count, dtype: int64
drawers
Yes they are solid plastic                     248
No there are no drawers or most are missing     30
Yes they are made from plastic coated wire      19
Name: count, dtype: int64


In [28]:
# Check whether all freezer variables are non-missing after cleaning
freezer_variables = ["size_freezer_1", "size_freezer_2", "temp_freezer", "icing", "refrigerant", "drawers"]
for var in freezer_variables:
    assert equipment.loc[equipment["equipment"] == "freezer", var].isna().sum() == 0, f"Variable {var} still has missing values for freezer after cleaning."


### (1e) ULT variables (ult_type, temp_ult, size_ult, seals, spacing, filter)

In [ ]:
# ULT type
# possible options: (1) Stirling Engine - <10yrs, HC
#                   (2) Dual compressor - Any Age, HC
#                   (3) Cascade compressor - <10yrs, HC
#                   (4) Cascade compressor - >10yrs, HC
#                   (5) Cascade compressor - Any Age, HFC
#                   (6) Dual compressor - <10yrs, HFC
#                   (7) Dual compressor - Any Age, HC

# Check unique values of ult_type
print(equipment[equipment["equipment"] == "ult"]["ult_type"].unique())
print(equipment[equipment["equipment"] == "ult"]["ult_type"].value_counts(dropna=False))

# Replace "Unknown" or missing with mode
equipment = fill_with_equipment_mode(
    equipment,
    value_col="ult_type",
    equipment_value="ult"
)

print(equipment[equipment["equipment"] == "ult"]["ult_type"].value_counts(dropna=False))

# Create variable ult_type_1 which is the engine type (Dual compressors, Cascade compressors, Stirling Engine, Unknown)
equipment.loc[
    (equipment["equipment"] == "ult") &
    (equipment["ult_type"].str.contains("Dual compressor", na=False)),
    "ult_type_1"
] = "Dual compressors"
equipment.loc[
    (equipment["equipment"] == "ult") &
    (equipment["ult_type"].str.contains("Cascade", na=False)),
    "ult_type_1"
] = "Cascade compressors"
equipment.loc[
    (equipment["equipment"] == "ult") &
    (equipment["ult_type"].str.contains("Stirling", na=False)),
    "ult_type_1"
] = "Stirling Engine"
equipment.loc[
    (equipment["equipment"] == "ult") & 
    (equipment["ult_type"].str.contains("Unknown", na=False)), # Unknown becomes cascade compressor
    "ult_type_1"
 ] = "Cascade compressors" 

# Create variable ult_type_2 which is the age and refrigerant type
# Cascade can not have HFC and age
equipment.loc[
    (equipment["equipment"] == "ult") &
    (equipment["ult_type"].str.contains("hydrocarbon", na=False) & equipment["ult_type"].str.contains("under 10", na=False)),
    "ult_type_2"
] = "<10yrs, HC"
equipment.loc[
    (equipment["equipment"] == "ult") &
    (equipment["ult_type_1"] == "Stirling Engine"), # Stirling must be <10yrs, HC
    "ult_type_2"
] = "<10yrs, HC"
equipment.loc[
    (equipment["equipment"] == "ult") & 
    (equipment["ult_type"] == "Dual compressor with hydrocarbon refrigerants"), # Dual with HC is "Any Age, HC"
    "ult_type_2"
] = "Any Age, HC"





# ##Two type questions, one for engine and one for refridgerent + age. First type can limit the second type. Second type is classified as Age by Bell#ult_type splits into type, age takes the refrigerant
# has_ult_type_mask = equipment["equipment"].apply(lambda x: any(val in x for val in ("ult")))
# #Extract engine type, to be put in Type column, added to a new column
# print(equipment["ult_type"].unique())
# print("Engine response mode is " + equipment.loc[has_ult_type_mask, "ult_type"].mode().iloc[0]) #Most common answer is "Unknown"! can't use the function
# print(equipment["ult_type"].value_counts())
# #Most common known-type: "Cascade compressors with hydrocarbon refrigerants, under 10 years old"

# equipment["ult_type_1"] = np.where(
#     equipment["ult_type"].str.contains("Dual compressor", na=False), "Dual compressors",
#     np.where(equipment["ult_type"].str.contains("Cascade", na=False), "Cascade compressors",
#              np.where(equipment["ult_type"].str.contains("Stirling", na=False), "Stirling Engine",
#                       np.where(equipment["ult_type"].str.contains("Unknown", na=False), "Cascade compressors", #See above
#                                None)))
# )
# print("Engine type mode is " + equipment.loc[has_ult_type_mask, "ult_type_1"].mode().iloc[0])
# #Extract age and refrigerant type to put into the age column, added to a new column
# #TODO has redundancy. First deals with cases with clear matching, then deals with forced cases. Maybe swap, but those values are prevented
# equipment["ult_type_2"] = np.where(
#     (equipment["ult_type"].str.contains("hydrocarbon", na=False) & equipment["ult_type"].str.contains("under 10", na=False)), "<10yrs, HC",
#     np.where((equipment["ult_type"].str.contains("hydrocarbon", na=False) & equipment["ult_type"].str.contains("over 10", na=False)), ">10yrs, HC",
#              np.where((equipment["ult_type"].str.contains("hydrofluorocarbon", na=False) & equipment["ult_type"].str.contains("under 10", na=False)), "<10yrs, HFC",
#                       np.where((equipment["ult_type"].str.contains("hydrofluorocarbon", na=False) & equipment["ult_type"].str.contains("over 10", na=False)), ">10yrs, HFC",
#                                np.where((equipment["ult_type"].str.contains("Unknown", na=False) & equipment["ult_type"].str.contains("under 10", na=False)), "<10yrs, HC", #Matches website guideline
#                                         np.where(equipment["ult_type"].str.contains("Stirling", na=False), "<10yrs, HC", #'Stirling engine' will always be "<10yrs, HC"
#                                                  np.where((equipment["ult_type"].str.contains("Dual", na=False) & ~equipment["ult_type"].str.contains("years", na=False)), "Any Age, HC", #'Dual compressor with hydrocarbon refrigerants'
#                                                           np.where((equipment["ult_type"].str.contains("Cascade", na=False) & ~equipment["ult_type"].str.contains("years", na=False)), "Any Age, HFC", #'Unknown with hydrofluorocarbon refrigerants', 'Cascade compressors with hydrofluorocarbon refrigerants', -> Any Age, HFC
#                                                                    np.where(equipment["ult_type"].str.contains("Unknown", na=False), "<10yrs, HC",
#                                                                    None))))))))
#                                         # Cascade can not have HFC and age
#                                         #Dual can not have HC and age
#                                         #Stirling must be <10yrs, HC
# )



['Unknown' 'Dual compressor with hydrocarbon refrigerants'
 'Cascade compressors with hydrocarbon refrigerants, under 10 years old'
 'Stirling engine with hydrocarbon refrigerants'
 'Dual compressor with hydrofluorocarbon refrigerants, under 10 years old'
 'Dual compressor with hydrofluorocarbon refrigerants, over 10 years old'
 nan 'Unknown with hydrofluorocarbon refrigerants'
 'Cascade compressors with hydrocarbon refrigerants, over 10 years old'
 'Unknown with hydrocarbon refrigerants, under 10 years old'
 'Cascade compressors with hydrofluorocarbon refrigerants']
ult_type
Unknown                                                                    41
Cascade compressors with hydrocarbon refrigerants, under 10 years old      20
Dual compressor with hydrocarbon refrigerants                              16
Dual compressor with hydrofluorocarbon refrigerants, under 10 years old    12
Cascade compressors with hydrocarbon refrigerants, over 10 years old       10
Stirling engine with hydroc

In [ ]:
# Temp

# Check unique values for temp_ult
print(equipment[equipment["equipment"] == "ult"]["temp_ult"].unique())
print(equipment[equipment["equipment"] == "ult"]["temp_ult"].value_counts(dropna=False))

#Run set_temp function to move all points to nearest valid option
equipment["temp_ult"] = equipment["temp_ult"].apply(
    lambda t: assign_set_temp(t, [-70, -75, -80])
)
print(equipment["temp_ult"].unique())

#Size of ULT must be reformatted
#Must fold Small single door (<500L) into smallest category
print(equipment["size_ult"].unique())
equipment["size_ult"] = np.where(
    equipment["size_ult"].str.contains("500"), "500-600L Tall", #will cover both categories
    np.where(equipment["size_ult"].str.contains("700"), "700-800L Tall",
             np.where(equipment["size_ult"].str.contains("900"), "800-900L Tall",
                      equipment["size_ult"]))
)
print(equipment["size_ult"].unique())

In [ ]:
# Seals

# Check unique values for seals and distribution
print(equipment[equipment["equipment"] == "ult"]["seals"].unique())
print(equipment[equipment["equipment"] == "ult"]["seals"].value_counts(dropna = False))

# Match formatting and collapse two non-damaged responses
# Policy on how to replace categories that are the same but overlap?
print(equipment["seals"].unique())
equipment["seals"] = np.where(
    equipment["seals"].isin(["Dirty or lightly iced", "Damaged, absent in places, or with a layer of ice"]), "No, seals are intact",
    np.where(equipment["seals"].isin(["Clean and in good condition"]), "Yes, seals are damaged",
             equipment["seals"])
)
print(equipment["seals"].unique())

In [ ]:
# Spacing

# Check unique values for spacing and distribution
print(equipment[equipment["equipment"] == "ult"]["spacing"].unique())
print(equipment[equipment["equipment"] == "ult"]["spacing"].value_counts(dropna=False))

# >= 150/Little spacing -> Good/Little
# print(equipment["spacing"].unique())
#Only two options, so matching here is exact
print(equipment["spacing"].unique())
equipment["spacing"] = equipment["spacing"].replace({"There's little space around the unit and we store items on top of the unit.": "Little / no spacing",
                                                     "≥150mm gap at the back and sides of the unit, with no items stored on top.": "Good spacing around unit (>= 150 mm on all sides)"})
print(equipment["spacing"].unique())

In [ ]:
# Filter

# Check unique values 
print(equipment[equipment["equipment"] == "ult"]["filter"].unique())
print(equipment[equipment["equipment"] == "ult"]["filter"].value_counts(dropna=False))

#Fix filter formatting
print(equipment["filter"].unique())
equipment["filter"] = np.where(
    equipment["filter"].isin(["It's clear"]), "No, filters are clear", # removed , "It's a little dirty" from folding in
    np.where(equipment["filter"].isin(["Most of it is clogged"]), "Yes, filters are clogged",
             equipment["filter"])#Leave 'No filter' option
)
print(equipment["filter"].unique())

### (1f) Glassware variables

In [ ]:
# Clean Glassware Variables

#Capacity reformating, just dropping first character
print(equipment["capacity_glassware"].unique())
equipment["capacity_glassware"] = equipment["capacity_glassware"].str.split("~").str[1]
print(equipment["capacity_glassware"].unique())

#tech is now age and has simpler formatting
print(equipment["tech"].unique())
equipment["tech"] = np.where(
    equipment["tech"].str.contains("Modern", na=False), "Newer", 
    np.where(equipment["tech"].str.contains("older", na=False), "Older", 
             equipment["tech"])
)

#Map to correct options that match wording
print(equipment["fan"].unique())
equipment["fan"] = np.where(
    equipment["fan"].str.contains("Yes"), "Yes, has a fan",
    np.where(equipment["fan"].str.contains("No"), "No fan",
             equipment["fan"]) #Fan has no I don't know option
)

# Remove I don't know option
equipment["temp_glassware"] = np.where(
    equipment["temp_glassware"].isin(["I don't know"]), np.nan, equipment["temp_glassware"]
)
#Assign values to nearest valid temperature
equipment["temp_glassware"] = equipment["temp_glassware"].apply(
    lambda t: assign_set_temp(t, [50, 60, 75])
)


In [ ]:
# Clean MSC Variables

#Already is a numeric value
print(equipment["width"].unique())
#Remove invalid option
equipment["width"] = equipment["width"].replace({1600: 1500})

#Age values must match format
print(equipment["age_microbio"].unique())
equipment["age_microbio"] = equipment["age_microbio"].replace({"Less than 5 years old": "<5 yrs",
                                                               "5-20 years old": "5-20yrs",
                                                               "More than 20 years old": ">20yrs"})
#Replace I don't know for age with mode
fill_with_equipment_mode(
    equipment,
    value_col="age_microbio",
    equipment_value="microbio",
    unknown_value="I don't know"
)

# Match No, recirculating / Yes, its ducted
print(equipment["ducting"].unique())
equipment["ducting"] = equipment["ducting"].replace({"No": "No, recirculating",
                                                     "Yes": "Yes, its ducted"})


In [ ]:
# Clean Incubator Variables

#No <150L, Add to 150-170L
print(equipment["capacity_incubator"].unique())
equipment["capacity_incubator"] = equipment["capacity_incubator"].replace({"<150L": "150-170L",
                                                                           "~150-170L": "150-170L",
                                                                           "~220-260L": "220-260L"})

#For age, ><20 -> Newer/Older
print(equipment["age_incubator"].unique())
equipment["age_incubator"] = equipment["age_incubator"].replace({"≤ 20 years": "Newer",
                                                                  "> 20 years": "Older"})

In [ ]:
# Clean water bath variables

# Fill missing/Unknown values for capacity_bath, heating, temp_bath, lid
for col in ["capacity_bath", "heating", "temp_bath", "lid"]:
    equipment = fill_with_equipment_mode(
        equipment,
        value_col=col,
        equipment_value="bath",
    )

# Modify "capacity_bath" to combine "10-12L" and "6-10L" into "10L". Currently mapped based on distance to median
print(equipment["capacity_bath"].unique())
equipment["capacity_bath"] = equipment["capacity_bath"].replace({"10-12L": "10L",
                                                                 "6-10L": "10L",
                                                                 "Less than 5L": "5-6L"})


# Set temperature for water bath rows, assigning to nearest valid value
#85 -> 90,/ 42, 40, 49, 46, 22(?) -> 37, /51 -> ?, 56, 58 -> 65 (Data -> SPARKHub val)
print(equipment["temp_bath"].unique())
equipment["temp_bath"] = np.where(
    equipment["temp_bath"].isin(["42", "40", "49", "46", "22"]), "37",
    np.where(equipment["temp_bath"].isin(["56", "58"]), "65",  #TODO? what to do about 51, which was mean(37, 65) 51 will be split in cleaning
             np.where(equipment["temp_bath"].isin(["85"]), "90",
                      equipment["temp_bath"]))
)
#Make sure string words are cleaned first, then should work (there are no strings)
equipment["temp_bath"] = equipment["temp_bath"].apply(
    lambda t: assign_set_temp(t, [37, 65, 90]) #Currently, 51 is set to na
)
print(equipment["temp_bath"].unique())

#Fix formatting of lid and heating
print(equipment["lid"].unique())
equipment["lid"] = equipment["lid"].replace({"Yes": "Lid is on during use",
                                             "No": "Lid is off during use"})

print(equipment["heating"].unique())
equipment["heating"] = equipment["heating"].replace({"Metal beads": "Beads"})

In [ ]:
# Clean cryostat variables

# Fill missing/Unknown values for temp_cryostat and sleep_mode
for col in ["temp_cryostat", "sleep_mode"]:
    equipment = fill_with_equipment_mode(
        equipment,
        value_col=col,
        equipment_value="cryostat",
    )

# Print unique values in sleep_mode
print(equipment["sleep_mode"].unique())

# Modify sleep_mode variable with "Energy Saving Mode Available"/"No Energy Saving Mode Available" based on sleep_mode
equipment["sleep_mode"] = np.where(
    (equipment["sleep_mode"].isin([
        "Yes, we use the unit 8am to 5pm, outside these hours it's in sleep mode",
        "Yes, but it's not used",
        "Yes, we use the unit 9am to 5pm, outside these hours it's in sleep mode"
        ]) | equipment["sleep_mode"].isna())    
        & (equipment["equipment"] == "cryostat"),
    "Energy Saving Mode Available", #All these values are treated as showing energy-saving mode is available now
    np.where((equipment["sleep_mode"] == "No, it does not")
             & (equipment["equipment"] == "cryostat"),
             "No Energy Saving Mode Available",
             equipment["sleep_mode"])
)

# Set temperatures only for cryostat rows
print(equipment["temp_cryostat"].unique())
# cryostat_mask = equipment["equipment"] == "cryostat"
equipment["temp_cryostat"] = equipment["temp_cryostat"].apply(
    lambda t: assign_set_temp(t, [-25, -20, -18]) #ensures only SPARKHub values are in data
)


In [ ]:
# Clean Heater Variables

#Match format, currently includes word "block"
print(equipment["blocks"].unique())
equipment["blocks"]= equipment["blocks"].str.extract(r'^(\d+)') #Extracts the numeric characters

# # 10(?), 45, 50 -> 37 / 60, 61, 64, 66, 76 -> 65 / 90 -> 95 / 120 -> 100 / 51, 80 -> ?
print(equipment["temp_heater"].unique())
equipment["temp_heater"] = np.where(
    equipment["temp_heater"].isin(["10", "45", "50"]), "37", #TODO 10 will be recleaned
    np.where(equipment["temp_heater"].isin(["60", "61", "64", "66", "76"]), "65",  #TODO what to do about 51, which was mean(37, 65) will be split in cleaning
             np.where(equipment["temp_heater"].isin(["90"]), "95", #TODO what to do about 80? will be split in cleaning
                      np.where(equipment["temp_heater"].isin(["120"]), "100", #TODO 120 will be rechecked
                               equipment["temp_heater"])))
)
#TODO Replace above with this once above tasks are checked
# equipment["temp_heater"] = np.where(
#     equipment["temp_heater"].isin(["Unknown"]), np.nan, equipment["temp_heater"]
# )
# equipment["temp_heater"] = equipment["temp_heater"].apply(
#     lambda t: assign_set_temp(t, [37, 65, 95, 100]) #ensures only SPARKHub values are in data
# )

print(equipment["temp_heater"].unique())

In [ ]:
# Clean it variables

# Check unique values of it_type
print(equipment["it_type"].unique())

# Fill missing/Unknown values for it_type
equipment = fill_with_equipment_mode(
    equipment,
    value_col="it_type",
    equipment_value="it",
)

## Monitor variable split for size (with the other equipments) and brightness (its own column)
print(equipment["monitor"].unique())
#Extract size related information and match to accepted values into a new column
equipment["monitor_size"] = np.where(
    equipment["monitor"].str.contains("Small", na=False), "19-22 inches",
    np.where(equipment["monitor"].str.contains("Medium", na=False), "24-27 inches",
             np.where(equipment["monitor"].str.contains("Large", na=False), "30+ inches",
                      np.where(equipment["monitor"].str.contains("does not", na=False), "No monitor", #to allow for the actual computers
                               equipment["monitor"]))) #Set to monitor value
)
#Extract brightness related information and match to accepted values into a new column
equipment["monitor_brightness"] = np.where(
    equipment["monitor"].str.contains("lowest", na=False), "Lowest",
    np.where(equipment["monitor"].str.contains("mid", na=False), "Mid",
             np.where(equipment["monitor"].str.contains("full", na=False), "Full",
                      equipment["monitor"])) #Set to monitor value
)

print(equipment["monitor_size"].unique())
print(equipment["monitor_brightness"].unique())

#Case matching is incorrect for it_type
print(equipment["it_type"].unique())
equipment["it_type"] = equipment["it_type"].replace({'High-powered desktop computer with graphics card': 'High-Powered Desktop Computer with Graphics Card'})

In [ ]:
# Map all variables to their new names
print(equipment["equipment"].unique())

# 3. Create columns to match and force variable types
## Process which shifts all the relevant data to the same column name that Isobel used.
## Changes regarding correct exact entries to be made afterwards
## Each section has a function which ensures only the relevant equipment have their values from these columns selected
## Other equipment gets None as the value

# Type = [controller_type, first halt size_fridge, first halt size_freezer, first half of ult_type, sleep_mode, it_type]
def equipment_type_func(row):
    if row["equipment"] == "fc":
        return row["controller_type"]
    elif row["equipment"] == "fridge":
        return row["size_fridge_1"]
    elif row["equipment"] == "freezer":
        return row["size_freezer_1"]
    elif row["equipment"] == "ult":
        return row["ult_type_1"]
    elif row["equipment"] == "cryostat":
        return row["sleep_mode"]
    elif row["equipment"] == "it":
        return row["it_type"]
    else:
        return None
equipment["Type"] = equipment.apply(lambda row: equipment_type_func(row), axis=1)


# Work_Surface_Width = [sash_width, width]
def equipment_width_func(row):
    if row["equipment"] == "fc":
        return row["sash_width"]
    elif row["equipment"] == "microbio":
        return row["width"]
    else:
        return None
equipment["Work_Surface_Width"] = equipment.apply(lambda row: equipment_width_func(row), axis=1)

# Size = [second half size_fridge, second half size_freezer, second half size_ult, capacity_bath, capacity_incubator, first half monitor]
def equipment_size_func(row):
    if row["equipment"] == "fridge":
        return row["size_fridge_2"]
    elif row["equipment"] == "freezer":
        return row["size_freezer_2"]
    elif row["equipment"] == "ult":
        return row["size_ult"]
    elif row["equipment"] == "bath":
        return row["capacity_bath"]
    elif row["equipment"] == "incubator":
        return row["capacity_incubator"]
    elif row["equipment"] == "it":
        return row["monitor_size"]
    else:
        return None
equipment["Size"] = equipment.apply(lambda row: equipment_size_func(row), axis=1)

# N_blocks = [blocks]
equipment["N_blocks"] = equipment["blocks"]
equipment["N_blocks"] = pd.to_numeric(equipment["N_blocks"], errors='coerce')

# Age = [secold half of ult_type (with age), tech, age_microbio, age_incubator]
def equipment_age_func(row):
    if row["equipment"] == "ult":
        return row["ult_type_2"]
    elif row["equipment"] == "cryostat":
        return row["tech"]
    elif row["equipment"] == "microbio":
        return row["age_microbio"]
    elif row["equipment"] == "incubator":
        return row["age_incubator"]
    else:
        return None
equipment["Age"] = equipment.apply(lambda row: equipment_age_func(row), axis=1)

# Set_Temp = [temp_freezer, temp_ult, temp_glassware, temp_cryostat, temp_bath, temp_heater]
def equipment_temp_func(row):
    if row["equipment"] == "freezer":
        return row["temp_freezer"]
    elif row["equipment"] == "ult":
        return row["temp_ult"]
    elif row["equipment"] == "glassware":
        return row["temp_glassware"]
    elif row["equipment"] == "cryostat":
        return row["temp_cryostat"]
    elif row["equipment"] == "bath":
        return row["temp_bath"]
    elif row["equipment"] == "heater":
        return row["temp_heater"]
    else:
        return None
equipment["Set_Temp"] = equipment.apply(lambda row: equipment_temp_func(row), axis=1)
equipment["Set_Temp"] = pd.to_numeric(equipment["Set_Temp"], errors='coerce')

# Brightness = [second half monitor]
equipment["Brightness"] = equipment["monitor_brightness"]

# Penalty indicators = [icing, drawers, seals, spacing, filter, fan, ducting, heating, lid, refrigerant]

# fc_specific = [face_velocity, lifted, surface, ] TODO look at code for this information

In [ ]:
# print(equipment.columns.values)
# print(equipment.shape)
print(equipment.iloc[:9, 57:64].dtypes)
#Examine each new column individually for data verification
def column_checker(colname):
    print("Counts for " + colname + "\n")
    print(equipment[colname].dtype.name)
    print(equipment[colname].unique())

column_checker("Type")
#We have no computer screen types? -> First merge on IT type, then on on the size/brightness etc
column_checker("Work_Surface_Width")
column_checker("Size")
column_checker("N_blocks")
column_checker("Age")
column_checker("Set_Temp")
column_checker("Brightness")
# print(equipment["Type"])


## Clean up and save processed data

In [ ]:
# Save processed data
# equipment.to_csv(config.PROCESSED_DATA / "panel_processed_3.csv", index =False)